In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
# import numpy as np
from torch.utils.data import Dataset, DataLoader

In [ ]:
raw_text = "hello world! welcome to building a tiny language model from scratch. " * 50

chars = sorted(list(set(raw_text)))
print("|".join(chars))
print(len(chars))

In [ ]:
vocab_size = len(chars)
# Here instead of creating manual dict we can use tokenizer
char_to_idx = {ch: i for i,ch in enumerate(chars)}
idx_to_char = {i: ch for i,ch in enumerate(chars)}

def encode(s):
    return [char_to_idx[c] for c in s]

def decode(l):
    return ''.join([idx_to_char[i] for i in l])

tokens = encode(raw_text)
print(tokens[:10])
print(decode(tokens[:10]))

In [ ]:
class TextDataset(Dataset):
  def __init__(self, data , block_size) -> None:
     super().__init__()
     self.data = data
     self.block_size = block_size
  
  def __len__(self):
    return len(self.data) - self.block_size
  
  def __getitem__(self, index):
    x = self.data[index : index + self.block_size]
    y = self.data[index + 1 : index + self.block_size + 1]
    return torch.tensor(x) , torch.tensor(y)
  
dataset = TextDataset(tokens, 4)
print(dataset.__getitem__(0))

In [ ]:
class MiniLLm(nn.Module):
   """A minimal language model with token embeddings, positional encodings, and a GRU layer."""
   def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        # Recurrent layer to capture sequential context across historical dependencies
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True, num_layers=1)
        self.lm_head = nn.Linear(hidden_dim, vocab_size)

   def forward(self, idx):
        # idx shape: (batch_size, sequence_length)
        x = self.token_embedding(idx)       # Out: (batch_size, sequence_length, embed_dim)
        out, _ = self.rnn(x)               # Out: (batch_size, sequence_length, hidden_dim)
        logits = self.lm_head(out)          # Out: (batch_size, sequence_length, vocab_size)
        return logits


def generate_text(model, start_str, max_new_tokens, device):
  model.eval()
  generated_indices = encode(start_str)
  input_tensor = torch.tensor(generated_indices, dtype=torch.long, device=device)

  with torch.no_grad():
    for _ in range(max_new_tokens):
      logits = model(input_tensor.unsqueeze(0))#shape: C,

      logits = logits[:, -1, :]# for last token

      probs = F.softmax(logits, dim=-1)

      next_idx = torch.multinomial(probs, num_samples=1).squeeze(0)

      generated_indices.append(next_idx.item())
      input_tensor = torch.cat([input_tensor, next_idx], dim=0)
    return "".join(idx_to_char[i] for i in generated_indices)




In [ ]:
from torch.types import Device
# training loop
if __name__ == "__main__":
    # Hyperparameters
    BATCH_SIZE = 32
    BLOCK_SIZE = 16  # Maximum contextual length sequence handled per sample window
    EMBED_DIM = 64
    HIDDEN_DIM = 128
    LEARNING_RATE = 0.005
    EPOCHS = 10
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


    dataset = TextDataset(tokens, BLOCK_SIZE)
    data_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    model = MiniLLm(vocab_size, EMBED_DIM, HIDDEN_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    print(f"Starting training on device : {DEVICE}")
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for batch_idx, (x, y) in enumerate(data_loader):
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
          
        average_loss = total_loss / (batch_idx + 1)
        print(f"Epoch {epoch + 1}/{EPOCHS}, Average Loss: {average_loss:.4f}")
    print("-"*10 + "Training Completed!" + "-"*10)

prompt = "hello "
print(f"--- Prompt: '{prompt}' ---")
print(f"Generated text output:\n{generate_text(model, prompt, max_new_tokens=60, device=DEVICE)}")

In [ ]:
prompt = input("Enter the prompt:")
print(f"--- Prompt: '{prompt}' ---")
generated_text = generate_text(model, prompt, max_new_tokens=100, device=DEVICE)
print(f"Generated text output:\n{generated_text}")
print(len(generated_text))